I want to compare different checkpoints during training, and also try out some inference-time modifications.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# General
from typing import Tuple, Sequence, Dict, Union, Optional
import numpy as np
import json
import torch
import torch.nn as nn
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter
import gerry
import pickle

# diffusion policy import
from diffusers.schedulers.scheduling_ddpm import DDPMScheduler
from diffusers.training_utils import EMAModel
from diffusers.optimization import get_scheduler

# Painting imports
import cv2
from style.diffusion_policy_gml.dataset import GmlDatasetNoSliding
from style.diffusion_policy_gml.network import compute_noise, compute_orig
import style.diffusion_policy_gml.network as network
from style.diffusion_policy_gml.network_transformer import Transformer1d, Transpose
import style.diffusion_policy_gml.utils as utils

In [ ]:
RUN_FOLDER = Path('runs/Jul08_01-18-24_eagle')

with open(RUN_FOLDER / 'network_kwargs.json', 'r') as f:
    network_kwargs_prelim = json.load(f)
PRED_HORIZON = network_kwargs_prelim['horizon']
PREDICTION_DIM = network_kwargs_prelim['output_dim']
NUM_DIFFUSION_ITERS = 100
device = torch.device('cuda')

print(f'{PRED_HORIZON=}, {PREDICTION_DIM=}, {NUM_DIFFUSION_ITERS=}')

## Load Dataset
(for normalization/un-normalization)

In [ ]:
# dataset_path = "data/gml_by_drawing_PRESERVE_ASPECT_CENTERED_003000.zarr"
dataset_path = "data/gml_by_stroke_PRESERVE_ASPECT_CENTERED_003000.zarr"

with gerry.Stopwatch("Loading dataset"):
    dataset = GmlDatasetNoSliding(
        dataset_path=dataset_path,
        sequence_length=PRED_HORIZON,
        action_delta=True,
        action_penlift=True,
        ignore_jump_actions=True,
        normalize=dict(obs=False, action=True),
        prescale_obs = (-3, 3),
        prescale_penlift=(0, 3),
        max_drawings=1,
        min_traj_length=15
    )
utils.drawing_lims = dict(x=(-3, 3), y=(-3, 3))

## Diffusion Setup

In [ ]:
# Noise scheduler
noise_scheduler = DDPMScheduler(
    num_train_timesteps=NUM_DIFFUSION_ITERS,
    beta_schedule='squaredcos_cap_v2',
    clip_sample=True,
    clip_sample_range=3,
    prediction_type='epsilon'
)

In [ ]:
# Load the network_kwargs
n_emb = 768
InputEmbedding = nn.Sequential(
    Transpose(1, 2),
    nn.Conv1d(3, 15, 5, padding=2),
    nn.ReLU(),
    nn.Conv1d(15, n_emb, 5, padding=2),
    nn.ReLU(),
    Transpose(1, 2)
)

_, network_kwargs = Transformer1d.FromJson(RUN_FOLDER / 'network_kwargs.json', InputEmbedding)
for k, v in network_kwargs.items():
    print(f'{k:<20}', v)

In [ ]:
def load_model(checkpoint=None):
    fname = f'ema_noise_pred_net_{checkpoint}.pth' if checkpoint is not None else 'ema_noise_pred_net.pth'
    ema_noise_pred_net = Transformer1d(**network_kwargs)
    ema_noise_pred_net.to(device)
    ema_noise_pred_net.load_state_dict(torch.load(RUN_FOLDER / fname))
    return ema_noise_pred_net

ema_noise_pred_net = load_model()

## Inference

In [ ]:
# Full generation
B = 7  # num samples

def generate(ema_noise_pred_net, B=5, history=None):
    x_init = torch.randn((B, PRED_HORIZON, PREDICTION_DIM), device=device)

    x_out = network.eval(ema_noise_pred_net, noise_scheduler, x_init,
                            global_cond=None,
                            log_history=history)

    x_out = x_out.detach().cpu().numpy()
    obs = dataset.unnormalize_obs(x_out[..., :2])
    penup = x_out[..., 2]

    x_out_ = utils.kill_after_penlift(x_out, -1)
    obs_ = dataset.unnormalize_obs(x_out_[..., :2])
    penup_ = x_out_[..., 2]

    return obs, penup, obs_, penup_

history = []
obs, penup, obs_, penup_ = generate(ema_noise_pred_net, B, history)

In [ ]:
# Plot history

to_plots = history[::10]
B_ = 3
fig, axes = plt.subplots(B_, len(to_plots), figsize=(20, 6), sharey=True)
for i, traj in enumerate(to_plots):
    t = i / len(to_plots)
    x_out_ = utils.kill_after_penlift(traj, -1)
    for b in range(B_):
        utils.plot_traj(axes[b][i], x_out_[b, :, 2], obs=x_out_[b, :, :2])

In [ ]:
# Plot drawings

axes = None
for b in range(obs.shape[0]):
    fig, axes = utils.plot_result(dataset, penup[b], obs[b], clean=True, axes=axes, traj_line_kwargs=dict(alpha=0.5));
fig.suptitle(f'{network_kwargs["n_layer"]} layers, {network_kwargs["n_head"]} heads, {network_kwargs["n_emb"]} emb', fontsize=24)

# Compare checkpoints

In [ ]:
checkpoints = RUN_FOLDER.glob('ema_noise_pred_net_*.pth')
checkpoints = [f.name.split('_')[-1].split('.')[0] for f in checkpoints]
checkpoints = sorted(map(int, checkpoints))
print(checkpoints)
print(len(checkpoints))

B = 15
all_results = {}
for checkpoint in tqdm(checkpoints):
    ema_noise_pred_net_ = load_model(checkpoint)
    all_results[checkpoint] = generate(ema_noise_pred_net_, B)

In [ ]:
R, C = 5, 6
fig, axes = plt.subplots(R, C, figsize=(20, 15), sharex=True, sharey=True)

for i, (ax, checkpoint) in enumerate(zip(axes.flatten(), checkpoints)):
    obs, penup, obs_, penup_ = all_results[checkpoint]

    for b in range(B):
        # utils.plot_result(dataset, penup[b], obs[b], clean=True, axes=ax, traj_line_kwargs=dict(alpha=0.5))
        utils.plot_traj(ax, penup_[b], obs=obs_[b])
    ax.set_title(f'Checkpoint {checkpoint}')

fig.suptitle(f'Overnight training transformer context=128 (Jul08_01-18-24_eagle)', fontsize=24, y=0.99)
fig.tight_layout()

folder = Path('results/gerry13z/')
folder.mkdir(exist_ok=True, parents=True)
plt.savefig(folder / 'Jul08_01-18-24_eagle.svg')
pickle.dump(all_results, open(folder / 'Jul08_01-18-24_eagle.pkl', 'wb'))